# Byte I/O - Rust

All 20 Rust examples from [docs/core/io.md](https://platob.github.io/yggdryl/core/io/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::io::{Buffer, IOBase};

let mut handle = Buffer::new();
handle.pwrite(0, b"symbol,price\n")?;
handle.pwrite(13, b"AAPL,1\n")?;
assert_eq!(handle.size(), 20);

// Two reads at different offsets, in any order: there is no shared cursor.
let mut tail = [0_u8; 4];
handle.pread(13, &mut tail)?;
let mut head = [0_u8; 6];
handle.pread(0, &mut head)?;
assert_eq!(&head, b"symbol");
assert_eq!(&tail, b"AAPL");

## Laziness

In [ ]:
use yggdryl::io::IOBase;
use yggdryl::{IOKind, local};

let path = std::env::temp_dir().join("yggdryl-docs-io-lazy.csv");
let _ = std::fs::remove_file(&path);

// Constructing touches nothing: no file is created, opened, or mapped.
let mut handle = local::File::new(&path)?;
assert!(!handle.exists());

// Reading something absent yields nothing rather than failing.
assert_eq!(handle.size(), 0);
let mut probe = [0_u8; 8];
assert_eq!(handle.pread(0, &mut probe)?, 0);
assert_eq!(handle.kind(), IOKind::Unknown);

// Writing creates the resource, and any parent it needs.
handle.write_all_bytes(b"symbol,price\n")?;
assert_eq!(handle.kind(), IOKind::File);
assert_eq!(handle.read_all()?, b"symbol,price\n");

handle.close()?;
std::fs::remove_file(&path)?;

## Kinds

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::{IOKind, local};

assert_eq!(Buffer::new().kind(), IOKind::Memory);
assert!(IOKind::Memory.is_leaf());

let folder = local::Folder::new(std::env::temp_dir())?;
assert_eq!(folder.kind(), IOKind::Directory);
assert!(folder.is_container());

// Nothing is there, so nothing has decided; a write settles it.
let absent = local::File::new(std::env::temp_dir().join("yggdryl-docs-io-absent.bin"))?;
assert_eq!(absent.kind(), IOKind::Unknown);
assert!(!absent.kind().is_known());

## Whole values

In [ ]:
use yggdryl::io::{Buffer, IOBase};

let mut handle = Buffer::new();
handle.write_all_bytes(b"symbol,price\n")?;

// `append` reports the offset the bytes landed at.
assert_eq!(handle.append(b"AAPL,1\n")?, 13);
assert_eq!(handle.read_range(0, 6)?, b"symbol");
// A range past the end yields what exists rather than failing.
assert!(handle.read_range(100, 4)?.is_empty());
assert_eq!(handle.read_all()?.len(), 20);

## Streaming adapters

In [ ]:
use std::io::{Read, Write};

use yggdryl::io::{Buffer, IOBase};

let mut handle = Buffer::new();
handle.writer_at(0).write_all(b"symbol,price\n")?;
handle.append(b"AAPL,1\n")?;

let mut text = String::new();
handle.reader_at(13).read_to_string(&mut text)?;
assert_eq!(text, "AAPL,1\n");

## What the bytes are

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::MimeType;

// Nothing names an in-memory buffer, so its type comes from its bytes.
let mut handle = Buffer::from_bytes(br#"{"symbol":"AAPL"}"#.to_vec());
assert_eq!(handle.media_type().base(), &MimeType::JSON);

// It is re-derived after the content changes.
handle.write_all_bytes(b"PAR1payload")?;
assert_eq!(handle.media_type().base(), &MimeType::PARQUET);

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::{Codec, MimeType, Url};

// A declared type wins, and the codings it carries are what `codec` reports.
let named = Buffer::new().with_media_type(Url::from_str("file:///trades.json.gz")?.media_type());
assert_eq!(named.media_type().base(), &MimeType::JSON);
assert_eq!(named.codec(), Codec::Gzip);

## Open and close

In [ ]:
use yggdryl::io::{Buffer, Coded, IOBase};
use yggdryl::Codec;

let mut handle = Coded::new(Buffer::new(), Codec::Zstd);
assert!(!handle.is_open());

handle.open()?;
assert!(handle.is_open());
handle.write_all_bytes(b"symbol,price\n")?;

// Closing publishes the pending write and releases the cache.
handle.close()?;
assert!(!handle.is_open());

// The handle stays usable; the next read re-materializes.
assert_eq!(handle.read_all()?, b"symbol,price\n");

## Buffer

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::MimeType;

let mut handle = Buffer::with_capacity(1_024);
handle.reserve(4_096)?;
assert!(handle.capacity() >= 4_096);
// Reserving changes the allocation, never the length.
assert_eq!(handle.size(), 0);

handle.pwrite(0, b"symbol,price\n")?;
assert_eq!(handle.as_slice(), b"symbol,price\n");

// A format the bytes cannot identify is declared rather than guessed.
let csv = Buffer::from_bytes(handle.into_bytes()).with_media_type(MimeType::CSV.into());
assert_eq!(csv.media_type().base(), &MimeType::CSV);

## Coded

In [ ]:
use yggdryl::io::{Buffer, Coded, IOBase};
use yggdryl::{Codec, Level, MimeType, Url};

let inner = Buffer::new().with_media_type(Url::from_str("file:///trades.arrows.gz")?.media_type());
let mut handle = Coded::new(inner, Codec::Gzip).with_level(Level::BEST);

// The wrapper's bytes are decoded, so its media type has the coding removed.
assert_eq!(handle.media_type().base(), &MimeType::ARROW_STREAM);
assert_eq!(handle.media_type().encoding_len(), 0);

let payload = "symbol,price\n".repeat(64).into_bytes();
handle.write_all_bytes(&payload)?;
handle.flush()?;

// Reads decompress; the wrapped handle only ever holds the encoded form.
assert_eq!(handle.read_all()?, payload);
assert!(handle.handle().size() < payload.len() as u64);

## Roles

In [ ]:
use yggdryl::io::IOBase;
use yggdryl::{IOKind, MimeType, local};

let path = std::env::temp_dir().join("yggdryl-docs-io-folder");
let _ = std::fs::remove_dir_all(&path);
let mut folder = local::Folder::new(&path)?;

// A container holds no bytes: reads are empty, byte writes are refused.
let mut probe = [0_u8; 4];
assert_eq!(folder.pread(0, &mut probe)?, 0);
assert_eq!(folder.size(), 0);
let refused = folder.pwrite(0, b"x").unwrap_err().to_string();
assert!(refused.contains("got the directory"), "{refused}");

// Truncating to zero is the write that brings a container into being.
folder.truncate(0)?;
assert!(folder.exists());
assert_eq!(folder.kind(), IOKind::Directory);
assert_eq!(folder.media_type().base(), &MimeType::DIRECTORY);
assert!(folder.ls(false, false)?.is_empty());

std::fs::remove_dir_all(&path)?;

In [ ]:
use yggdryl::io::IOBase;
use yggdryl::{IOKind, local};

// A location that arrived from outside answers by looking at what is there.
let existing = local::Path::new(std::env::temp_dir())?;
assert_eq!(existing.kind(), IOKind::Directory);

let undecided = local::Path::new(std::env::temp_dir().join("yggdryl-docs-io-undecided"))?;
assert_eq!(undecided.kind(), IOKind::Unknown);
assert!(undecided.read_all()?.is_empty());

// A leaf is not a container: it lists nothing and resolves no child.
let leaf = local::File::new(std::env::temp_dir().join("yggdryl-docs-io-leaf.arrows"))?;
assert!(leaf.ls(true, false)?.is_empty());
assert!(leaf.child_by("nested").is_err());

## Delegating to a wrapped handle

In [ ]:
use yggdryl::io::{Buffer, IOBase};

/// A wrapper mirrors the handle's bytes rather than owning bytes of its own.
struct Counted {
    handle: Buffer,
    opens: usize,
}

impl IOBase for Counted {
    yggdryl::delegate_iobase!(handle);

    fn open(&mut self) -> yggdryl::Result<()> {
        self.opens += 1;
        self.handle.open()
    }
}

fn main() -> Result<(), Box<dyn std::error::Error>> {
    let mut wrapper = Counted {
        handle: Buffer::new(),
        opens: 0,
    };
    wrapper.open()?;
    wrapper.write_all_bytes(b"AAPL")?;

    assert_eq!(wrapper.opens, 1);
    assert_eq!(wrapper.read_all()?, b"AAPL");
    assert_eq!(wrapper.handle.as_slice(), b"AAPL");
    Ok(())
}

## Arrow batches

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use yggdryl::arrow;
use yggdryl::io::{Buffer, IOBase};
use yggdryl::{DataType, Url};

// A non-null struct Field is the schema.
let schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("symbol"),
])?
.required_field("row");

let arrow_schema = schema.to_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![
        Arc::new(Int64Array::from(vec![1, 2])),
        Arc::new(StringArray::from(vec![Some("AAPL"), None])),
    ],
)?;

// The handle's own media type picks the encoding; no format argument is passed.
let mut handle = Buffer::new().with_media_type(Url::from_str("file:///trades.arrows")?.media_type());
let options = handle.record_options()?;

// The write path takes a batch reader and nothing else.
handle.write_arrow_batch_reader(arrow::batch_reader(arrow_schema, [batch]), &options)?;
assert_eq!(handle.read_arrow_field(&options)?, schema);

// The read path returns one. Batches arrive one at a time, never as a vector.
let mut rows = 0;
for batch in handle.read_arrow_batch_reader(&options)? {
    rows += batch?.num_rows();
}
assert_eq!(rows, 2);

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::MimeType;

// An absent resource holds no batches rather than failing to parse.
let empty = Buffer::new().with_media_type(MimeType::ARROW_STREAM.into());
assert_eq!(
    empty.read_arrow_batch_reader(&empty.record_options()?)?.count(),
    0
);

// An encoding this build does not implement is named rather than guessed.
let csv = Buffer::new().with_media_type(MimeType::CSV.into());
let message = csv.record_options().unwrap_err().to_string();
assert!(message.contains("text/csv"), "{message}");

## Column pushdown

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, RecordBatchReader, StringArray};
use yggdryl::arrow;
use yggdryl::generic::IORecordOptions;
use yggdryl::io::{Buffer, IOBase};
use yggdryl::{DataType, MimeType};

let stored = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.required_field("symbol"),
    DataType::Utf8.required_field("venue"),
])?
.required_field("row");
let arrow_schema = stored.to_arrow_schema()?;

let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![
        Arc::new(Int64Array::from(vec![1, 2])),
        Arc::new(StringArray::from(vec!["AAPL", "MSFT"])),
        Arc::new(StringArray::from(vec!["XNAS", "XNAS"])),
    ],
)?;

let mut handle = Buffer::new().with_media_type(MimeType::ARROW_STREAM.into());
let plain = handle.record_options()?;
handle.write_arrow_batch_reader(arrow::batch_reader(arrow_schema, [batch]), &plain)?;

// One of the three columns, declared as this read's schema.
let wanted = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");

let projected = handle.read_arrow_batch_reader(&plain.clone().with_schema(wanted))?;
assert_eq!(projected.schema().fields().len(), 1);
assert_eq!(projected.map(|batch| batch.unwrap().num_columns()).sum::<usize>(), 1);

// The resource is unchanged: it still holds all three.
assert_eq!(handle.read_arrow_field(&plain)?.field_len(), 3);

// A column it does not hold cannot be projected out of it, so the encoding
// reads everything and the cast supplies that column as nulls.
let invented = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("nowhere"),
])?
.required_field("row");
let widened = handle.read_arrow_batch_reader(&plain.with_schema(invented))?;
assert_eq!(widened.schema().fields().len(), 2);
assert_eq!(widened.schema().field(1).name(), "nowhere");

## Appending and merging

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use yggdryl::arrow;
use yggdryl::generic::IORecordOptions;
use yggdryl::io::{Buffer, IOBase};
use yggdryl::{DataType, Url};

let schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("symbol"),
])?
.required_field("row");
let arrow_schema = schema.to_arrow_schema()?;
let rows = |ids: Vec<i64>, symbols: Vec<&'static str>| {
    let batch = RecordBatch::try_new(
        Arc::clone(&arrow_schema),
        vec![
            Arc::new(Int64Array::from(ids)),
            Arc::new(StringArray::from(symbols)),
        ],
    )
    .expect("a batch matching the root");
    arrow::batch_reader(batch.schema(), [batch])
};

let mut handle =
    Buffer::new().with_media_type(Url::from_str("file:///trades.arrows")?.media_type());
let options = handle.record_options()?.with_schema(schema.clone());

// No match key: the resource is replaced.
handle.write_arrow_batch_reader(rows(vec![1, 2], vec!["AAPL", "MSFT"]), &options)?;

// Appending reads what is there, chains the new batches after it, and rewrites.
handle.append_arrow_batch_reader(rows(vec![3], vec!["NVDA"]), &options)?;
let total: usize = handle
    .read_arrow_batch_reader(&options)?
    .map(|batch| batch.unwrap().num_rows())
    .sum();
assert_eq!(total, 3);

// A match key merges: `2` is already stored and updates, `9` is new and appends.
let merging = options.clone().with_merge_by(["id"]);
handle.write_arrow_batch_reader(rows(vec![2, 9], vec!["MSFT.O", "AMD"]), &merging)?;
let total: usize = handle
    .read_arrow_batch_reader(&options)?
    .map(|batch| batch.unwrap().num_rows())
    .sum();
assert_eq!(total, 4);

## Globbing and Hive partitions

In [ ]:
use yggdryl::io::IOBase;
use yggdryl::local::Folder;

let root = std::env::temp_dir().join("yggdryl-doc-lake");
let _ = std::fs::remove_dir_all(&root);
for year in ["2024", "2025"] {
    let leaf = root.join(format!("year={year}")).join("month=01");
    std::fs::create_dir_all(&leaf)?;
    std::fs::write(leaf.join("part-0.parquet"), b"parquet")?;
}

let lake = Folder::new(&root)?;

// A fixed prefix is descended, not listed and filtered.
assert_eq!(lake.glob("year=2024/**/*.parquet", false)?.len(), 1);
assert_eq!(lake.glob("**/*.parquet", false)?.len(), 2);

// Partition filters select the leaves to overwrite or upsert.
let selected: Vec<_> = lake.children_where(&[("year", "2024")], false)?.collect();
assert_eq!(selected.len(), 1);
assert_eq!(selected[0].partitions(), vec![
    ("year".to_owned(), "2024".to_owned()),
    ("month".to_owned(), "01".to_owned()),
]);

let _ = std::fs::remove_dir_all(&root);

## Partition columns in the data

In [ ]:
use yggdryl::generic::{Holder, IORecordOptions, RecordOptions};
use yggdryl::io::IOBase;
use yggdryl::{DataType, MimeType};

let root = std::env::temp_dir().join("yggdryl-doc-partitioned");
let _ = std::fs::remove_dir_all(&root);
std::fs::create_dir_all(root.join("year=2024").join("month=01"))?;

let schema = DataType::from_fields([
    DataType::Int64.required_field("price"),
    DataType::Int32.required_field("year"),
    DataType::Utf8.required_field("month"),
])?
.required_field("row");
let arrow_schema = schema.to_arrow_schema()?;
let batch = arrow_array::RecordBatch::try_new(
    std::sync::Arc::clone(&arrow_schema),
    vec![
        std::sync::Arc::new(arrow_array::Int64Array::from(vec![10, 20])),
        std::sync::Arc::new(arrow_array::Int32Array::from(vec![2024, 2024])),
        std::sync::Arc::new(arrow_array::StringArray::from(vec!["01", "01"])),
    ],
)?;

// The rows carry every column; the write drops the two the path spells out.
let mut lake = Holder::folder(&root)?;
let options = RecordOptions::for_mime_type(&MimeType::ARROW_STREAM)?.with_schema(schema.clone());
lake.write_arrow_batch_reader(
    yggdryl::arrow::batch_reader(arrow_schema, [batch]),
    &options,
)?;

// Only `price` reached the leaf; the other two are the directory names.
let leaf = lake.child_by("year=2024/month=01/part-0.arrows")?;
assert_eq!(
    leaf.read_arrow_field(&RecordOptions::for_media_type(leaf.media_type())?)?.field_len(),
    1
);

// Reading the folder restores them with their declared types.
let restored = lake
    .read_arrow_batch_reader(&options)?
    .next()
    .expect("one batch")?;
assert_eq!(restored.num_columns(), 3);
assert_eq!(restored.schema().field(1).data_type(), &arrow_schema::DataType::Int32);

let _ = std::fs::remove_dir_all(&root);

In [ ]:
use yggdryl::generic::{Holder, IORecordOptions, RecordOptions};
use yggdryl::io::IOBase;
use yggdryl::{DataType, MimeType};

let root = std::env::temp_dir().join("yggdryl-doc-declared-layout");
let _ = std::fs::remove_dir_all(&root);
std::fs::create_dir_all(&root)?;

// Nothing is on disk, so nothing spells a layout. The schema does.
let schema = DataType::from_fields([
    DataType::Int64.required_field("price"),
    DataType::Int32.required_field("year"),
])?
.required_field("row")
.with_partition_fields(&["year"])?;
assert_eq!(schema.partition_field_names().collect::<Vec<_>>(), ["year"]);

let arrow_schema = schema.to_arrow_schema()?;
let batch = arrow_array::RecordBatch::try_new(
    std::sync::Arc::clone(&arrow_schema),
    vec![
        std::sync::Arc::new(arrow_array::Int64Array::from(vec![10, 20])),
        std::sync::Arc::new(arrow_array::Int32Array::from(vec![2024, 2024])),
    ],
)?;

let mut lake = Holder::folder(&root)?;
let options = RecordOptions::for_mime_type(&MimeType::ARROW_STREAM)?.with_schema(schema);
lake.write_arrow_batch_reader(
    yggdryl::arrow::batch_reader(arrow_schema, [batch]),
    &options,
)?;

// The directory came from the declaration, and the leaf stores what the
// path does not carry.
assert!(root.join("year=2024").is_dir());

// Reading it back reports the layout without being told it.
let derived = lake.read_arrow_field(
    &RecordOptions::for_mime_type(&MimeType::ARROW_STREAM)?,
)?;
assert_eq!(derived.partition_field_names().collect::<Vec<_>>(), ["year"]);

let _ = std::fs::remove_dir_all(&root);